In [2]:
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torchvision.utils as vutils
from PIL import Image  
from torchvision import datasets, transforms
import torch.nn.functional as F
from torchvision.utils import save_image
from math import log2, sqrt
from tqdm import tqdm

In [3]:
# Constants
DATA_DIR = 'data'
SAVE_PATH = 'preprocessed_data_forGAN.npz'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = (64, 64)
IMG_DIR_NAMES = ['wiki','wiki_augmented', 'inpainting', 'insight', 'text2img']
REAL_IMG_DIR = 'wiki'

## Functions to calculate the metrics

In [4]:
import torch.nn.functional as F
from pytorch_fid import fid_score
from torchvision.models import inception_v3

# Create directories for storing images for FID calculation
if not os.path.exists('fid_real'):
    os.makedirs('fid_real')
if not os.path.exists('fid_generated'):
    os.makedirs('fid_generated')

# 1. FID Score Implementation
def save_real_images(dataloader, num_images=1000):
    """Save real images for FID calculation"""
    count = 0
    for i, (data, _) in enumerate(dataloader):
        for j in range(data.shape[0]):
            if count >= num_images:
                break
            img = data[j].cpu().detach().permute(1, 2, 0).numpy()
            # Convert from [-1, 1] to [0, 255]
            img = ((img + 1) * 127.5).astype(np.uint8)
            Image.fromarray(img).save(f'fid_real/real_{count}.png')
            count += 1
        if count >= num_images:
            break

def save_generated_images(generator, latent_dim, device, num_images=1000, batch_size=64):
    """Generate and save images for FID calculation"""
    count = 0
    generator.eval()
    with torch.no_grad():
        while count < num_images:
            curr_batch_size = min(batch_size, num_images - count)
            noise = torch.randn(curr_batch_size, latent_dim, 1, 1, device=device)
            fake = generator(noise).cpu().detach()
            
            for j in range(fake.shape[0]):
                img = fake[j].permute(1, 2, 0).numpy()
                # Convert from [-1, 1] to [0, 255]
                img = ((img + 1) * 127.5).astype(np.uint8)
                Image.fromarray(img).save(f'fid_generated/fake_{count}.png')
                count += 1
                if count >= num_images:
                    break

def calculate_fid(device):
    """Calculate FID score between real and generated images"""
    try:
        fid_value = fid_score.calculate_fid_given_paths(
            ['fid_real/', 'fid_generated/'],
            batch_size=50,
            device=device,
            dims=2048)
        return fid_value
    except Exception as e:
        print(f"FID calculation error: {e}")
        return float('nan')

# 2. Inception Score Implementation
def inception_score(images, device, splits=10):
    """Calculate Inception Score for a tensor of images"""
    # Load inception model
    try:
        inception_model = inception_v3(pretrained=True).to(device)
        inception_model.eval()
        up = nn.Upsample(size=(299, 299), mode='bilinear', align_corners=True).to(device)
        print("Inception model loaded.")

        # Resize images to inception input size (299x299)
        resized_images = up(images)

        # Get predictions
        preds = []
        with torch.no_grad():
            pred = F.softmax(inception_model(resized_images), dim=1)
            preds.append(pred.cpu().numpy())

        preds = np.concatenate(preds, axis=0)
        num_images = preds.shape[0]

        # Calculate inception score
        scores = []
        for i in range(splits):
            part = preds[i * (num_images // splits): (i + 1) * (num_images // splits), :]
            kl = part * (np.log(part) - np.log(np.mean(part, axis=0, keepdims=True)))
            kl = np.mean(np.sum(kl, axis=1))
            scores.append(np.exp(kl))

        return np.mean(scores), np.std(scores)
    except Exception as e:
        print(f"Inception score calculation error: {e}")
        return float('nan'), float('nan')

# 3. Discriminator Accuracy Tracker
class AccuracyTracker:
    def __init__(self):
        self.real_correct = 0
        self.fake_correct = 0
        self.real_total = 0
        self.fake_total = 0
        
    def update_real(self, predictions, target_is_real=True):
        """Update accuracy on real images"""
        target = 1.0 if target_is_real else 0.0
        predictions = (predictions > 0.5).float()
        correct = (predictions == target).float().sum().item()
        self.real_correct += correct
        self.real_total += predictions.size(0)
        
    def update_fake(self, predictions, target_is_real=False):
        """Update accuracy on fake images"""
        target = 1.0 if target_is_real else 0.0
        predictions = (predictions > 0.5).float()
        correct = (predictions == target).float().sum().item()
        self.fake_correct += correct
        self.fake_total += predictions.size(0)
        
    def get_metrics(self):
        """Get the accuracy metrics"""
        real_acc = self.real_correct / max(self.real_total, 1)
        fake_acc = self.fake_correct / max(self.fake_total, 1)
        overall_acc = (self.real_correct + self.fake_correct) / max((self.real_total + self.fake_total), 1)
        return {
            'real_accuracy': real_acc,
            'fake_accuracy': fake_acc,
            'overall_accuracy': overall_acc
        }
        
    def reset(self):
        """Reset the counters for a new epoch"""
        self.real_correct = 0
        self.fake_correct = 0
        self.real_total = 0
        self.fake_total = 0

In [5]:
def isImgFake(img_main_dir):
    return int(img_main_dir != REAL_IMG_DIR)

## Preprocess the images

In [6]:
# Loading of the images and preprocessing
def load_and_preprocess_images(data_dir, img_size, save_path):
    """
    Loads and preprocesses images from the specified directory.
    Saves the preprocessed data to a file to avoid reprocessing.
    """
    # Check if preprocessed data already exists
    if os.path.exists(save_path):
        print(f"Loading preprocessed data from {save_path}...")
        data = np.load(save_path)
        return data['X'], data['y']
    
    print(f"Loading and preprocessing images from {data_dir}...")
    X = []
    y = []
    
    for index, main_folder in enumerate(IMG_DIR_NAMES):
        main_folder_path = os.path.join(data_dir, main_folder)
        print(f"Processing directory {main_folder}...")
        image_count = 0
        for subfolder_name in os.listdir(main_folder_path):
            subfolder_path = os.path.join(main_folder_path, subfolder_name)
            if os.path.isdir(subfolder_path) and main_folder == 'wiki':
                for filename in os.listdir(subfolder_path):
                    if filename.endswith('.jpg'):
                        img_path = os.path.join(subfolder_path, filename)
                        try:
                            img = Image.open(img_path).convert('RGB')  # Ensure RGB format
                            img = img.resize(img_size)
                            img_array = (np.array(img) / 127.5) - 1.0  # Normalize to [-1, 1]
                            X.append(img_array)
                            y.append(isImgFake(main_folder))  # 1 for other, 0 for wiki
                            image_count += 1
                        except Exception as e:
                            print(f"Error processing {img_path}: {e}")
                    else:
                        print(f"Skipping non jpg file: {filename}")
                        
        print(f"  - Loaded {image_count} images for category {main_folder}")

    print(f"Total images loaded: {len(X)}")
    
    # Convert to numpy arrays
    X = np.array(X)
    y = np.array(y)
    
    print(X)
    print(y)
    
    # Save preprocessed data
    print(f"Saving preprocessed data to {save_path}...")
    np.savez(save_path, X=X, y=y)
    
    return X, y

In [7]:
print("\n--- Starting Image Classification ---")

# Ensure the data directory exists
if not os.path.exists(DATA_DIR):
	os.makedirs(DATA_DIR)
	for dir_name in IMG_DIR_NAMES:
		os.makedirs(os.path.join(DATA_DIR, dir_name))

# Load and preprocess data
X, y = load_and_preprocess_images(DATA_DIR, IMG_SIZE, SAVE_PATH)

# Split the data
print("\nSplitting data into training and testing sets (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)  # Change shape to [batch, channels, height, width]
X_test = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)  # Reshape for BCE loss
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


--- Starting Image Classification ---
Loading preprocessed data from preprocessed_data_forGAN.npz...

Splitting data into training and testing sets (80/20)...


In [8]:
DEVICE                  = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS                  = 50
LEARNING_RATE           = 1e-3
BATCH_SIZE              = 64
LOG_RESOLUTION          = 6  # for 64x64 images (2^6 = 64)
Z_DIM                   = 64
W_DIM                   = 64
LAMBDA_GP               = 10

In [9]:
# Create necessary directories
if not os.path.exists('saved_examples'):
    os.makedirs('saved_examples')
if not os.path.exists('models'):
    os.makedirs('models')

## Functions related to the StyleGAN implementation

In [10]:
def get_loader():
    """Create a DataLoader using your preprocessed dataset"""
    # Create DataLoaders
    train_dataset = TensorDataset(X_train, y_train)
    batch_size = BATCH_SIZE
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,
        num_workers=2,
        pin_memory=True if DEVICE == "cuda" else False
    )
    return train_loader

In [11]:
class EqualizedWeight(nn.Module):

    def __init__(self, shape):

        super().__init__()

        self.c = 1 / sqrt(np.prod(shape[1:]))
        self.weight = nn.Parameter(torch.randn(shape))

    def forward(self):
        return self.weight * self.c

In [12]:
class Conv2dWeightModulate(nn.Module):

    def __init__(self, in_features, out_features, kernel_size,
                 demodulate = True, eps = 1e-8):

        super().__init__()
        self.out_features = out_features
        self.demodulate = demodulate
        self.padding = (kernel_size - 1) // 2

        self.weight = EqualizedWeight([out_features, in_features, kernel_size, kernel_size])
        self.eps = eps

    def forward(self, x, s):

        b, _, h, w = x.shape

        s = s[:, None, :, None, None]
        weights = self.weight()[None, :, :, :, :]
        weights = weights * s

        if self.demodulate:
            sigma_inv = torch.rsqrt((weights ** 2).sum(dim=(2, 3, 4), keepdim=True) + self.eps)
            weights = weights * sigma_inv

        x = x.reshape(1, -1, h, w)

        _, _, *ws = weights.shape
        weights = weights.reshape(b * self.out_features, *ws)

        x = F.conv2d(x, weights, padding=self.padding, groups=b)

        return x.reshape(-1, self.out_features, h, w)


In [13]:
class EqualizedConv2d(nn.Module):

    def __init__(self, in_features, out_features,
                 kernel_size, padding = 0):

        super().__init__()
        self.padding = padding
        self.weight = EqualizedWeight([out_features, in_features, kernel_size, kernel_size])
        self.bias = nn.Parameter(torch.ones(out_features))

    def forward(self, x: torch.Tensor):
        return F.conv2d(x, self.weight(), bias=self.bias, padding=self.padding)

In [14]:
class EqualizedLinear(nn.Module):

    def __init__(self, in_features, out_features, bias = 0.):

        super().__init__()
        self.weight = EqualizedWeight([out_features, in_features])
        self.bias = nn.Parameter(torch.ones(out_features) * bias)

    def forward(self, x: torch.Tensor):
        return F.linear(x, self.weight(), bias=self.bias)

class MappingNetwork(nn.Module):
    def __init__(self, z_dim, w_dim):
        super().__init__()
        self.mapping = nn.Sequential(
            EqualizedLinear(z_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(w_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(w_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(w_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(w_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(w_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(w_dim, w_dim),
            nn.ReLU(),
            EqualizedLinear(w_dim, w_dim)
        )

    def forward(self, x):
        x = x / torch.sqrt(torch.mean(x ** 2, dim=1, keepdim=True) + 1e-8)  # for PixelNorm 
        return self.mapping(x)

In [15]:
class StyleBlock(nn.Module):

    def __init__(self, W_DIM, in_features, out_features):

        super().__init__()

        self.to_style = EqualizedLinear(W_DIM, in_features, bias=1.0)
        self.conv = Conv2dWeightModulate(in_features, out_features, kernel_size=3)
        self.scale_noise = nn.Parameter(torch.zeros(1))
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.activation = nn.LeakyReLU(0.2, True)

    def forward(self, x, w, noise):

        s = self.to_style(w)
        x = self.conv(x, s)
        if noise is not None:
            x = x + self.scale_noise[None, :, None, None] * noise
        return self.activation(x + self.bias[None, :, None, None])

In [16]:
class ToRGB(nn.Module):

    def __init__(self, W_DIM, features):

        super().__init__()
        self.to_style = EqualizedLinear(W_DIM, features, bias=1.0)

        self.conv = Conv2dWeightModulate(features, 3, kernel_size=1, demodulate=False)
        self.bias = nn.Parameter(torch.zeros(3))
        self.activation = nn.LeakyReLU(0.2, True)

    def forward(self, x, w):

        style = self.to_style(w)
        x = self.conv(x, style)
        return self.activation(x + self.bias[None, :, None, None])

In [17]:
class GeneratorBlock(nn.Module):

    def __init__(self, W_DIM, in_features, out_features):

        super().__init__()

        self.style_block1 = StyleBlock(W_DIM, in_features, out_features)
        self.style_block2 = StyleBlock(W_DIM, out_features, out_features)

        self.to_rgb = ToRGB(W_DIM, out_features)

    def forward(self, x, w, noise):

        x = self.style_block1(x, w, noise[0])
        x = self.style_block2(x, w, noise[1])

        rgb = self.to_rgb(x, w)

        return x, rgb


In [18]:
class Generator(nn.Module):

    def __init__(self, log_resolution, W_DIM, n_features = 32, max_features = 256):

        super().__init__()

        features = [min(max_features, n_features * (2 ** i)) for i in range(log_resolution - 2, -1, -1)]
        self.n_blocks = len(features)

        self.initial_constant = nn.Parameter(torch.randn((1, features[0], 4, 4)))

        self.style_block = StyleBlock(W_DIM, features[0], features[0])
        self.to_rgb = ToRGB(W_DIM, features[0])

        blocks = [GeneratorBlock(W_DIM, features[i - 1], features[i]) for i in range(1, self.n_blocks)]
        self.blocks = nn.ModuleList(blocks)

    def forward(self, w, input_noise):

        batch_size = w.shape[1]

        x = self.initial_constant.expand(batch_size, -1, -1, -1)
        x = self.style_block(x, w[0], input_noise[0][1])
        rgb = self.to_rgb(x, w[0])

        for i in range(1, self.n_blocks):
            x = F.interpolate(x, scale_factor=2, mode="bilinear")
            x, rgb_new = self.blocks[i - 1](x, w[i], input_noise[i])
            rgb = F.interpolate(rgb, scale_factor=2, mode="bilinear") + rgb_new

        return torch.tanh(rgb)

In [19]:
class DiscriminatorBlock(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()
        self.residual = nn.Sequential(nn.AvgPool2d(kernel_size=2, stride=2), # down sampling using avg pool
                                      EqualizedConv2d(in_features, out_features, kernel_size=1))

        self.block = nn.Sequential(
            EqualizedConv2d(in_features, in_features, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, True),
            EqualizedConv2d(in_features, out_features, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, True),
        )

        self.down_sample = nn.AvgPool2d(
            kernel_size=2, stride=2
        )  # down sampling using avg pool

        self.scale = 1 / sqrt(2)

    def forward(self, x):
        residual = self.residual(x)

        x = self.block(x)
        x = self.down_sample(x)

        return (x + residual) * self.scale

In [20]:
class Discriminator(nn.Module):

    def __init__(self, log_resolution, n_features = 64, max_features = 256):

        super().__init__()

        features = [min(max_features, n_features * (2 ** i)) for i in range(log_resolution - 1)]

        self.from_rgb = nn.Sequential(
            EqualizedConv2d(3, n_features, 1),
            nn.LeakyReLU(0.2, True),
        )
        n_blocks = len(features) - 1
        blocks = [DiscriminatorBlock(features[i], features[i + 1]) for i in range(n_blocks)]
        self.blocks = nn.Sequential(*blocks)

        final_features = features[-1] + 1
        self.conv = EqualizedConv2d(final_features, final_features, 3)
        self.final = EqualizedLinear(2 * 2 * final_features, 1)

    def minibatch_std(self, x):
        batch_statistics = (
            torch.std(x, dim=0).mean().repeat(x.shape[0], 1, x.shape[2], x.shape[3])
        )
        return torch.cat([x, batch_statistics], dim=1)

    def forward(self, x):

        x = self.from_rgb(x)
        x = self.blocks(x)

        x = self.minibatch_std(x)
        x = self.conv(x)
        x = x.reshape(x.shape[0], -1)
        return self.final(x)

In [21]:
class PathLengthPenalty(nn.Module):

    def __init__(self, beta):

        super().__init__()

        self.beta = beta
        self.steps = nn.Parameter(torch.tensor(0.), requires_grad=False)

        self.exp_sum_a = nn.Parameter(torch.tensor(0.), requires_grad=False)

    def forward(self, w, x):

        device = x.device
        image_size = x.shape[2] * x.shape[3]
        y = torch.randn(x.shape, device=device)

        output = (x * y).sum() / sqrt(image_size)
        sqrt(image_size)

        gradients, *_ = torch.autograd.grad(outputs=output,
                                            inputs=w,
                                            grad_outputs=torch.ones(output.shape, device=device),
                                            create_graph=True)

        norm = (gradients ** 2).sum(dim=2).mean(dim=1).sqrt()

        if self.steps > 0:

            a = self.exp_sum_a / (1 - self.beta ** self.steps)

            loss = torch.mean((norm - a) ** 2)
        else:
            loss = norm.new_tensor(0)

        mean = norm.mean().detach()
        self.exp_sum_a.mul_(self.beta).add_(mean, alpha=1 - self.beta)
        self.steps.add_(1.)

        return loss

In [22]:
loader              = get_loader()

gen                 = Generator(LOG_RESOLUTION, W_DIM).to(DEVICE)
critic              = Discriminator(LOG_RESOLUTION).to(DEVICE)
mapping_network     = MappingNetwork(Z_DIM, W_DIM).to(DEVICE)
path_length_penalty = PathLengthPenalty(0.99).to(DEVICE)

opt_gen             = optim.Adam(gen.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.99))
opt_critic          = optim.Adam(critic.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.99))
opt_mapping_network = optim.Adam(mapping_network.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.99))

gen.train()
critic.train()
mapping_network.train()

MappingNetwork(
  (mapping): Sequential(
    (0): EqualizedLinear(
      (weight): EqualizedWeight()
    )
    (1): ReLU()
    (2): EqualizedLinear(
      (weight): EqualizedWeight()
    )
    (3): ReLU()
    (4): EqualizedLinear(
      (weight): EqualizedWeight()
    )
    (5): ReLU()
    (6): EqualizedLinear(
      (weight): EqualizedWeight()
    )
    (7): ReLU()
    (8): EqualizedLinear(
      (weight): EqualizedWeight()
    )
    (9): ReLU()
    (10): EqualizedLinear(
      (weight): EqualizedWeight()
    )
    (11): ReLU()
    (12): EqualizedLinear(
      (weight): EqualizedWeight()
    )
    (13): ReLU()
    (14): EqualizedLinear(
      (weight): EqualizedWeight()
    )
  )
)

## Utils

In [23]:
def gradient_penalty(critic, real, fake,device="cpu"):
    BATCH_SIZE, C, H, W = real.shape
    beta = torch.rand((BATCH_SIZE, 1, 1, 1)).repeat(1, C, H, W).to(device)
    interpolated_images = real * beta + fake.detach() * (1 - beta)
    interpolated_images.requires_grad_(True)

    # Calculate critic scores
    mixed_scores = critic(interpolated_images)
 
    # Take the gradient of the scores with respect to the images
    gradient = torch.autograd.grad(
        inputs=interpolated_images,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradient = gradient.view(gradient.shape[0], -1)
    gradient_norm = gradient.norm(2, dim=1)
    gradient_penalty = torch.mean((gradient_norm - 1) ** 2)
    return gradient_penalty

def get_w(batch_size):

    z = torch.randn(batch_size, W_DIM).to(DEVICE)
    w = mapping_network(z)
    return w[None, :, :].expand(LOG_RESOLUTION, -1, -1)

def get_noise(batch_size):
    
        noise = []
        resolution = 4

        for i in range(LOG_RESOLUTION):
            if i == 0:
                n1 = None
            else:
                n1 = torch.randn(batch_size, 1, resolution, resolution, device=DEVICE)
            n2 = torch.randn(batch_size, 1, resolution, resolution, device=DEVICE)

            noise.append((n1, n2))

            resolution *= 2

        return noise

def generate_examples(gen, epoch, n=100):
    
    gen.eval()
    alpha = 1.0
    for i in range(n):
        with torch.no_grad():
            w     = get_w(1)
            noise = get_noise(1)
            img = gen(w, noise)
            if not os.path.exists(f'saved_examples/epoch{epoch}'):
                os.makedirs(f'saved_examples/epoch{epoch}')
            save_image(img*0.5+0.5, f"saved_examples/epoch{epoch}/img_{i}.png")

    gen.train()

In [24]:
# Initialize metrics trackers
fid_scores = []
is_scores = []
disc_accuracies = []
epochs_measured = []
accuracy_tracker = AccuracyTracker()

In [25]:
# Modify the train_fn function to track discriminator accuracy
def train_fn(
    critic, gen, path_length_penalty, loader, opt_critic, opt_gen, opt_mapping_network, accuracy_tracker
):
    loop = tqdm(loader, leave=True)
    
    for batch_idx, (real, _) in enumerate(loop):
        real = real.to(DEVICE)
        cur_batch_size = real.shape[0]

        w = get_w(cur_batch_size)
        noise = get_noise(cur_batch_size)
        
        with torch.cuda.amp.autocast():
            fake = gen(w, noise)
            critic_fake = critic(fake.detach())
            critic_real = critic(real)
            
            # Update accuracy tracker
            accuracy_tracker.update_fake(torch.sigmoid(critic_fake))
            accuracy_tracker.update_real(torch.sigmoid(critic_real))
            
            gp = gradient_penalty(critic, real, fake, device=DEVICE)
            loss_critic = (
                -(torch.mean(critic_real) - torch.mean(critic_fake))
                + LAMBDA_GP * gp
                + (0.001 * torch.mean(critic_real ** 2))
            )

        critic.zero_grad()
        loss_critic.backward()
        opt_critic.step()

        gen_fake = critic(fake)
        loss_gen = -torch.mean(gen_fake)

        if batch_idx % 16 == 0:
            plp = path_length_penalty(w, fake)
            if not torch.isnan(plp):
                loss_gen = loss_gen + plp

        mapping_network.zero_grad()
        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()
        opt_mapping_network.step()

        loop.set_postfix(
            gp=gp.item(),
            loss_critic=loss_critic.item(),
        )

In [26]:
loader = get_loader()  

# Update the main training loop to calculate metrics
for epoch in range(EPOCHS):
    # Reset accuracy tracker for this epoch
    accuracy_tracker.reset()
    
    train_fn(
        critic,
        gen,
        path_length_penalty,
        loader,
        opt_critic,
        opt_gen,
        opt_mapping_network,
        accuracy_tracker
    )
    
    # Generate samples for visualization
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        with torch.no_grad():
            fixed_w = get_w(64)  
            fixed_noise = get_noise(64)
            fake = gen(fixed_w, fixed_noise).detach().cpu()
        
        # Save images
        if not os.path.exists('./results_styleGAN'):
            os.makedirs('./results_styleGAN')
        img_name = f'fake_samples_epoch_{epoch:03d}.png'
        img_path = os.path.join('./results_styleGAN', img_name)
        vutils.save_image(fake, img_path, normalize=True)
        
        print(f"Saved fake images for epoch {epoch} at {img_path}")

        # Calculate metrics every 5 epochs
        print(f"Epoch {epoch}: Calculating metrics...")
        
        # Get discriminator accuracy
        disc_metrics = accuracy_tracker.get_metrics()
        disc_accuracies.append(disc_metrics['overall_accuracy'])
        
        # Save real and fake images for FID
        save_real_images(loader, num_images=500)  # Using your existing function
        
        # Create function to save StyleGAN images
        def save_stylegan_images_for_fid(gen, num_images=500, batch_size=64):
            count = 0
            gen.eval()
            with torch.no_grad():
                while count < num_images:
                    curr_batch_size = min(batch_size, num_images - count)
                    w = get_w(curr_batch_size)
                    noise = get_noise(curr_batch_size)
                    fake = gen(w, noise).cpu().detach()
                    
                    for j in range(fake.shape[0]):
                        img = fake[j].permute(1, 2, 0).numpy()
                        # Convert from [-1, 1] to [0, 255]
                        img = ((img + 1) * 127.5).astype(np.uint8)
                        Image.fromarray(img).save(f'fid_generated/fake_{count}.png')
                        count += 1
                        if count >= num_images:
                            break
            gen.train()

        save_stylegan_images_for_fid(gen)
        
        # Calculate FID
        fid = calculate_fid(DEVICE)
        fid_scores.append(fid)
        
        # Calculate Inception Score with StyleGAN
        def calculate_is_for_stylegan(gen, num_images=500, batch_size=64):
            gen.eval()
            try:
                # Create batches of images
                fake_images_list = []
                with torch.no_grad():
                    for i in range(0, num_images, batch_size):
                        curr_batch_size = min(batch_size, num_images - i)
                        w = get_w(curr_batch_size)
                        noise = get_noise(curr_batch_size)
                        fake = gen(w, noise)
                        fake_images_list.append(fake)

                fake_images = torch.cat(fake_images_list, dim=0).to(DEVICE)
                is_mean, is_std = inception_score(fake_images, DEVICE, splits=10)
                return is_mean, is_std
            except Exception as e:
                print(f"IS calculation error: {e}")
                return float('nan'), float('nan')
            finally:
                gen.train()
        
        is_mean, is_std = calculate_is_for_stylegan(gen)
        is_scores.append(is_mean)
        
        epochs_measured.append(epoch)
        
        print(f"FID: {fid:.4f} | Inception Score: {is_mean:.4f}±{is_std:.4f}")
        print(f"Discriminator Accuracy: {disc_metrics['overall_accuracy']:.4f} " + 
              f"(Real: {disc_metrics['real_accuracy']:.4f}, Fake: {disc_metrics['fake_accuracy']:.4f})")
        
        # Save models
        torch.save({
            'gen': gen.state_dict(),
            'critic': critic.state_dict(),
            'mapping': mapping_network.state_dict(),
            'opt_gen': opt_gen.state_dict(),
            'opt_critic': opt_critic.state_dict(),
            'opt_mapping': opt_mapping_network.state_dict(),
            'epoch': epoch,
        }, f'models/stylegan_checkpoint_{epoch}.pt')

  0%|          | 0/375 [00:00<?, ?it/s]C:\Users\jmlim\AppData\Local\Temp\ipykernel_16408\1327616729.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\jmlim\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\autograd\graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
  7%|▋         | 27/375 [00:17<03:45,  1.54it/s, gp=0.663, loss_critic=4.53]


KeyboardInterrupt: 

In [ ]:
# Plot metrics at the end of training
plt.figure(figsize=(15, 5))

# Plot FID scores
plt.subplot(1, 3, 1)
plt.plot(epochs_measured, fid_scores, marker='o')
plt.title('FID Score (lower is better)')
plt.xlabel('Epoch')
plt.ylabel('FID')

# Plot Inception scores
plt.subplot(1, 3, 2)
plt.plot(epochs_measured, is_scores, marker='o')
plt.title('Inception Score (higher is better)')
plt.xlabel('Epoch')
plt.ylabel('IS')

# Plot Discriminator accuracy
plt.subplot(1, 3, 3)
plt.plot(epochs_measured, disc_accuracies)
plt.title('Discriminator Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.savefig('stylegan_metrics_plot.png')
plt.show()